In [1]:
from datasets import load_dataset
from tokenizers import (
    decoders,
    models,
    normalizers,
    pre_tokenizers,
    processors,
    trainers,
    Tokenizer,
)

In [4]:
dataset = load_dataset("wikitext", name="wikitext-2-raw-v1", split="train")

def get_training_corpus():
    for i in range(0, len(dataset), 1000):
        yield dataset[i: i+1000]["text"]

# BPE tokenizer
tokenizer = Tokenizer(models.BPE())

# pre-tokenizer - can use options like WhiteSpace()
# Ġ - this represents a white-space character
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
print(f"Tokenized sentence: {tokenizer.pre_tokenizer.pre_tokenize_str("Let's test pre-tokenization!")}")


# Define trainer
trainer = trainers.BpeTrainer(vocab_size=25000, special_tokens=["<|endoftext|>", "[IMG]", "[UNK]"])
tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)

# Encode string
encoding = tokenizer.encode("Let's test the tokenizer")
print(f"Tokens: {encoding.tokens}")
print(f"Output id: {encoding.ids}")

# Define decoder
tokenizer.decoder = decoders.ByteLevel()
print(f"\nDecoded ids: {tokenizer.decode(encoding.ids)}")

Tokenized sentence: [('ĠLet', (0, 3)), ("'s", (3, 5)), ('Ġtest', (5, 10)), ('Ġpre', (10, 14)), ('-', (14, 15)), ('tokenization', (15, 27)), ('!', (27, 28))]



Tokens: ['ĠLet', "'", 's', 'Ġtest', 'Ġthe', 'Ġto', 'ken', 'izer']
Output id: [8718, 9, 85, 2861, 200, 229, 3565, 11901]

Decoded ids:  Let's test the tokenizer


In [ ]:
# Decoding on custom dataset
tokenizer = Tokenizer(models.BPE())

tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

trainer = trainers.BpeTrainer(vocab_size=100)
tokenizer.train_from_iterator(['low', 'lower', 'lowest'], trainer=trainer)
print(f"Vocabulary learned: {tokenizer.get_vocab()}")

encoding = tokenizer.encode("lowering")
print(f"Tokens: {encoding.tokens}")
print(f"Output id: {encoding.ids}")

# The token IDs are dependent on the order characters are inserted into dictionary
# e gets 0 because e-w are inserted in alphabetical order.

# lowering is tokenized as: l o w e r i n g
# 1. lo
# 2. low
# 3. lowe
# 4. lower




Vocabulary learned: {'lower': 11, 'st': 10, 'r': 3, 'lowest': 12, 'lowe': 9, 'w': 6, 't': 5, 'o': 2, 'low': 8, 'l': 1, 'e': 0, 's': 4, 'lo': 7}
Tokens: ['lower']
Output id: [11]


In [ ]:
# Decoding on custom dataset - with special token
tokenizer = Tokenizer(models.BPE())

tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

trainer = trainers.BpeTrainer(vocab_size=100, special_tokens=["[UNK]"])
tokenizer.train_from_iterator(['low', 'lower', 'lowest'], trainer=trainer)
print(f"Vocabulary learned: {tokenizer.get_vocab()}")

encoding = tokenizer.encode("lowering the lowest load")
print(f"Tokens: {encoding.tokens}")
print(f"Output id: {encoding.ids}")

# The token IDs are dependent on the order characters are inserted into dictionary
# e gets 0 because e-w are inserted in alphabetical order.

# lowering is tokenized as: l o w e r i n g
# 1. lo
# 2. low
# 3. lowe
# 4. lower




Vocabulary learned: {'w': 7, 'lowe': 10, 's': 5, 'lowest': 13, 'o': 3, 'low': 9, 'l': 2, 't': 6, 'st': 11, 'e': 1, 'r': 4, 'lo': 8, '[UNK]': 0, 'lower': 12}
Tokens: ['lower', 't', 'e', 'lower', 'st', 'lo']
Output id: [12, 6, 1, 12, 11, 8]


### Questions

1. How does BPE work?
2. How does byte level BPE work?
3. What are those numbers in the ID?
4. What are normalizers? post processors?
5. How to use the template?
